In [1]:
import numpy as np
import seaborn as sns
import pandas as pd
from football_analytics.analyses.passing.features import passing_feature_columns

import os
os.environ["OMP_NUM_THREADS"] = "4"
from sklearn.preprocessing import StandardScaler

In [2]:
df_2018 = pd.read_parquet("../../../artifacts/features/player_level_wc2018.parquet")
df_2022 = pd.read_parquet("../../../artifacts/features/player_level_wc2022.parquet")

df_2018["dataset"] = "WC2018"
df_2022["dataset"] = "WC2022"

df = pd.concat([df_2018, df_2022], ignore_index=True)
df = df[df["player_position"] != "Goalkeeper"]
passing_cols = passing_feature_columns()
passing_df = df[["player_key", "player_position", "player"] + passing_cols]
orig_df = df.copy()

In [3]:
df = passing_df.copy()
df = df.dropna(subset=["player_position"])

In [4]:
FEATURES = [
    'passes_per_90','short_passes_per_90','medium_passes_per_90','long_passes_per_90',
    'crosses_per90','switches_per90','throughballs_per90','cutbacks_per90','backheels_per90',
    'passes_under_pressure_per90','key_passes_per90','assists_per90','pct_passes_under_pressure',
    'avg_pass_length','std_pass_length','pct_short_pass','pct_med_pass','pct_long_pass',
    'pct_ground_pass','pct_low_pass','pct_high_pass','pct_left_foot_pass','pct_right_foot_pass',
    'pct_head_pass','pct_foot_pass','pct_other_pass','pct_keeper_arm_pass',
    'pct_progressive_passes','pct_lateral_passes','pct_defensive_passes',
    'pct_pass_from_def_third','pct_pass_from_mid_third','pct_pass_from_att_third',
    'pct_pass_from_left_channel','pct_pass_from_central_channel','pct_pass_from_right_channel',
    'pct_pass_to_def_third','pct_pass_to_mid_third','pct_pass_to_left_channel',
    'pct_pass_to_central_channel','pct_pass_to_right_channel',
    'pct_passes_final_third','pct_passes_into_box',
    'pct_pass_from_zone_dl','pct_pass_to_zone_dl','pct_pass_from_zone_dc','pct_pass_to_zone_dc',
    'pct_pass_from_zone_dr','pct_pass_to_zone_dr','pct_pass_from_zone_ml','pct_pass_to_zone_ml',
    'pct_pass_from_zone_mc','pct_pass_to_zone_mc','pct_pass_from_zone_mr','pct_pass_to_zone_mr',
    'pct_pass_from_zone_al','pct_pass_to_zone_al','pct_pass_from_zone_ac','pct_pass_to_zone_ac',
    'pct_pass_from_zone_ar','pct_pass_to_zone_ar',
    'pct_pass_def_to_mid','pct_pass_def_to_att','pct_pass_mid_to_att','pct_pass_mid_to_mid',
    'pct_pass_att_to_mid','pct_pass_def_to_def','pct_pass_att_to_att',
    'pct_pass_left_to_centre','pct_pass_left_to_right','pct_pass_right_to_centre',
    'pct_pass_right_to_left','pct_pass_centre_to_left','pct_pass_centre_to_right',
    'pct_pass_wide_to_box','pct_pass_centre_to_box','pct_pass_def_to_box',
    'ttl_passes_F','pct_passes_F','passes_F_per90',
    'ttl_passes_FR','pct_passes_FR','passes_FR_per90',
    'ttl_passes_R','pct_passes_R','passes_R_per90',
    'ttl_passes_BR','pct_passes_BR','passes_BR_per90',
    'ttl_passes_B','pct_passes_B','passes_B_per90',
    'ttl_passes_BL','pct_passes_BL','passes_BL_per90',
    'ttl_passes_L','pct_passes_L','passes_L_per90',
    'ttl_passes_FL','pct_passes_FL','passes_FL_per90',
    'pass_angle_mean_overall','pass_angle_var_overall',
    'pass_angle_mean_def_third','pass_angle_var_def_third',
    'pass_angle_mean_mid_third','pass_angle_var_mid_third',
    'pass_angle_mean_att_third','pass_angle_var_att_third',
    'pass_angle_mean_left_channel','pass_angle_var_left_channel',
    'pass_angle_mean_centre_channel','pass_angle_var_centre_channel',
    'pass_angle_mean_right_channel','pass_angle_var_right_channel'
]

In [5]:
df0 = passing_df.copy()

df1 = df0[
    (df0["has_minutes"] == 1) &
    (df0["has_pass_events"] == 1) &
    (df0["passes_per_90"].fillna(0) > 0) &
    (df0["player_position"].notna())
].copy()

len(df0), len(df1), df1["player_position"].value_counts().head(30)

(1203,
 950,
 player_position
 Left Center Back             91
 Right Center Back            91
 Right Back                   84
 Center Forward               79
 Left Back                    73
 Right Wing                   61
 Left Wing                    60
 Right Center Midfield        57
 Left Center Midfield         56
 Right Defensive Midfield     43
 Center Attacking Midfield    36
 Left Defensive Midfield      35
 Center Defensive Midfield    27
 Right Midfield               27
 Left Midfield                27
 Right Center Forward         25
 Left Center Forward          21
 Center Back                  17
 Right Wing Back              16
 Left Wing Back               14
 Left Attacking Midfield       4
 Center Midfield               3
 Right Attacking Midfield      3
 Name: count, dtype: int64)

In [6]:
X = df1[FEATURES].copy()
X[FEATURES].isna().mean().sort_values(ascending=False).head(15)
X["backheels_per90"] = X["backheels_per90"].fillna(0.0)

In [7]:
df_model = X

In [8]:
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
X_scaled = scaler.fit_transform(df_model[FEATURES])
X_scaled = pd.DataFrame(X_scaled, columns=FEATURES, index=df_model.index)
X_scaled.head(10)

,passes_per_90,short_passes_per_90,medium_passes_per_90,long_passes_per_90,crosses_per90,switches_per90,throughballs_per90,cutbacks_per90,backheels_per90,passes_under_pressure_per90,...,pass_angle_mean_mid_third,pass_angle_var_mid_third,pass_angle_mean_att_third,pass_angle_var_att_third,pass_angle_mean_left_channel,pass_angle_var_left_channel,pass_angle_mean_centre_channel,pass_angle_var_centre_channel,pass_angle_mean_right_channel,pass_angle_var_right_channel
0,-0.627386,-0.175537,-0.637076,-0.683500,1.068761,-0.195181,0.0,0.000000,0.000000,0.465205,...,-0.649028,-0.044296,-0.480193,-0.159542,-0.652451,-2.034394,0.605813,-0.705338,-0.266882,0.082908
2,0.558188,0.889960,0.286026,0.527051,-0.077228,0.642161,0.0,0.000000,0.000000,1.898145,...,-0.395924,0.948948,-0.151062,0.414448,-2.198162,0.547231,0.109208,0.444732,0.127343,1.189872
3,-0.149580,0.109663,-0.196884,-0.112758,0.189568,-0.609397,0.0,0.000000,0.000000,1.586412,...,0.293110,0.166184,-0.366643,-0.053300,0.192261,0.014524,0.314958,0.392079,0.310721,0.310452
4,-0.439166,-0.159918,-0.551536,-0.127777,1.927500,0.825047,0.0,0.000000,0.316901,-0.506450,...,1.015235,-0.023467,0.818860,-0.363454,0.732574,-0.509431,1.798147,-1.121509,-0.429098,-0.851970
5,1.230567,0.752261,1.251846,1.451219,-0.344024,0.016382,0.0,0.000000,0.000000,0.131653,...,0.256233,-0.278989,0.118679,-0.526520,-0.221306,0.247753,-0.023961,-0.700349,1.336804,-1.920614
6,-0.458311,0.137735,-0.552489,-0.663735,0.702530,-0.404837,0.0,0.316344,2.530756,0.137497,...,-0.845088,-0.114265,-0.836999,0.726617,0.391094,0.248609,0.047145,0.408300,-0.738363,0.176888
7,0.030244,-0.499704,0.067515,1.008596,-0.344024,-0.082724,0.0,0.000000,0.000000,-0.186193,...,0.314963,-0.581035,0.083132,-1.866465,-0.002093,-0.003049,0.244739,-0.463934,1.379087,-1.337945
8,1.301344,-0.532935,1.113872,4.010453,-0.344024,4.396834,0.0,0.000000,0.000000,0.339476,...,-0.265472,-0.545294,0.006685,-1.866465,-0.400119,-0.774161,-0.383729,-0.591522,0.135296,0.136399
9,0.456516,0.761004,0.807262,-0.755951,-0.344024,-0.609397,0.0,0.000000,0.952381,0.926657,...,0.530302,0.723299,0.993838,0.443023,0.822950,0.773836,0.598564,0.470282,-0.958135,0.130696
10,0.021652,0.106296,-0.027030,0.230351,0.990609,0.386647,0.0,0.000000,0.220049,0.567115,...,-0.765661,0.757757,-0.631889,0.271190,0.134851,0.621402,-1.530614,0.732516,-0.385715,0.036383


In [17]:
from sklearn.decomposition import PCA

pca = PCA(n_components=20)
X_pca = pca.fit_transform(X_scaled)
X_pca = pd.DataFrame(
    X_pca,
    index=df_model.index,
    columns=[f"PC{i+1}" for i in range(X_pca.shape[1])]
)


In [19]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=9, random_state=42)
df_model["role_id"] = kmeans.fit_predict(X_pca.iloc[:, :20])
df_model["role_id"].value_counts()

role_id
3    196
7    163
8    130
4     97
2     93
1     91
5     85
0     65
6     30
Name: count, dtype: int64

In [22]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_std = pd.DataFrame(
    scaler.fit_transform(df_model[FEATURES]),
    columns=FEATURES,
    index=df_model.index
)

X_std["role_id"] = df_model["role_id"]
X_std.sample(10)

,passes_per_90,short_passes_per_90,medium_passes_per_90,long_passes_per_90,crosses_per90,switches_per90,throughballs_per90,cutbacks_per90,backheels_per90,passes_under_pressure_per90,...,pass_angle_var_mid_third,pass_angle_mean_att_third,pass_angle_var_att_third,pass_angle_mean_left_channel,pass_angle_var_left_channel,pass_angle_mean_centre_channel,pass_angle_var_centre_channel,pass_angle_mean_right_channel,pass_angle_var_right_channel,role_id
64,-0.747550,0.424423,-1.197437,-1.448049,-0.113848,-1.040409,-0.412331,4.236466,2.383575,0.350896,...,-1.211036,1.686912,-0.157820,1.288769,-0.052388,-2.023740,-0.279844,2.935915,-0.393517,6
56,0.503111,0.708590,0.277595,0.239704,-0.332771,-0.247387,-0.412331,-0.345530,1.435345,1.786522,...,0.831277,-0.174425,0.137489,-0.357360,0.697917,0.334754,0.448266,0.752063,0.796249,7
146,-1.015975,-0.775434,-1.032214,-0.692616,1.134763,1.355531,-0.412331,-0.345530,-0.364356,-0.092394,...,-0.771378,-0.461014,0.543954,0.909747,1.102450,-1.200268,-1.726852,-0.476022,0.343642,3
1121,2.205449,1.542065,2.400611,1.418043,-0.748278,0.768574,1.166281,0.681279,-0.364356,1.013588,...,1.010262,0.307304,0.633173,0.268927,0.704632,-0.039665,1.536150,-0.259401,0.245864,5
1079,-0.732327,-1.221097,-0.216085,-0.403775,-0.748278,-1.040409,-0.412331,-0.345530,-0.364356,-1.182633,...,-1.646569,0.023175,-2.181133,0.088049,-1.105863,0.295710,0.923599,-0.593540,-1.660932,2
978,1.296882,0.357088,1.556320,1.569903,-0.748278,-0.289681,-0.412331,-0.345530,-0.364356,0.415002,...,-0.764670,-0.656296,-2.181133,-0.468139,0.283032,-0.239591,-0.958864,0.133383,-1.970485,1
517,-0.133927,-0.713824,-0.147905,1.114191,-0.140010,0.507483,-0.412331,-0.345530,-0.364356,1.091095,...,0.725343,-0.227759,0.430625,-0.657068,0.146278,1.074612,0.986630,-0.227135,0.518131,7
958,-0.663708,-0.161628,-0.728965,-0.995175,-0.146217,-0.465872,2.094522,-0.345530,-0.364356,-0.609067,...,0.515541,0.993818,0.633327,0.803569,0.202273,0.453705,0.771200,-1.092132,0.636209,3
402,-1.035698,-0.943792,-0.855201,-0.851321,-0.500371,-0.882693,-0.412331,-0.345530,1.246301,-0.027548,...,1.717126,0.517415,1.285846,0.852013,1.295833,-0.460397,0.604308,-2.122656,-1.970485,3
694,-0.347721,0.080816,-0.507077,-0.560416,0.431761,-0.289681,-0.412331,-0.345530,-0.364356,-0.578649,...,0.654270,-1.114169,0.086441,1.232913,-0.114270,-1.154696,-0.528120,-0.901124,-0.359044,6


In [24]:
role_profiles = X_std.groupby("role_id")[FEATURES].mean()
for r in role_profiles.index:
    print(f"\nROLE {r}")
    print("Top + traits")
    print(role_profiles.loc[r].sort_values(ascending=False).head(10))
    print("\nTop - traits")
    print(role_profiles.loc[r].sort_values().head(10))



ROLE 0
Top + traits
pct_other_pass               1.314778
pct_pass_from_zone_ac        1.044481
pct_passes_final_third       0.916758
pct_pass_to_zone_ac          0.902429
pct_pass_att_to_att          0.881252
pct_pass_from_att_third      0.829776
pct_pass_centre_to_box       0.825180
pct_passes_under_pressure    0.802268
pass_angle_var_overall       0.793018
pct_short_pass               0.759793
Name: 0, dtype: float64

Top - traits
passes_F_per90            -0.768806
pct_pass_to_def_third     -0.742541
medium_passes_per_90      -0.739272
pct_pass_to_mid_third     -0.710769
pct_pass_from_def_third   -0.710592
long_passes_per_90        -0.699677
pct_med_pass              -0.692627
passes_per_90             -0.685386
avg_pass_length           -0.670090
pct_pass_def_to_def       -0.656363
Name: 0, dtype: float64

ROLE 1
Top + traits
pct_pass_from_zone_dl      2.079590
pct_pass_from_def_third    1.556646
pct_pass_to_zone_dc        1.481334
pct_pass_left_to_centre    1.479428
pct_pass_def

In [25]:
ROLE_TAXONOMY = {
    0: "Interior Attacking Midfielder (Advanced 8 / 10)",
    1: "Left Build-Up Centre Back",
    2: "Right Build-Up Centre Back",
    3: "Advanced Vertical Playmaker",
    4: "Left Attacking Wing Back",
    5: "Central Volume Distributor (Deep 6)",
    6: "Press-Relief Outlet",
    7: "Middle-Third Circulator (Box-to-Box 8)",
    8: "Right Attacking Wing Back",
}
df_model["role_name"] = df_model["role_id"].map(ROLE_TAXONOMY)



In [33]:
import hdbscan

hdb = hdbscan.HDBSCAN(min_cluster_size=30)
df_model["density_role"] = hdb.fit_predict(X_pca)
pd.crosstab(df_model["role_id"], df_model["density_role"])
df_model["density_role"].value_counts()

density_role
-1    950
Name: count, dtype: int64

In [29]:
from sklearn.metrics import pairwise_distances

centroids = pd.DataFrame(kmeans.cluster_centers_, columns=[f"pc{i}" for i in range(X_pca.shape[1])])
dist = pairwise_distances(X_pca, centroids, metric="cosine")
role_strength = pd.DataFrame(1 / (1 + dist), columns=[f"role_{i}_strength" for i in range(9)])
role_strength.head(10)

,role_0_strength,role_1_strength,role_2_strength,role_3_strength,role_4_strength,role_5_strength,role_6_strength,role_7_strength,role_8_strength
0,0.541027,0.380363,0.472122,0.669362,0.409524,0.419636,0.507292,0.452132,0.754517
1,0.462044,0.439337,0.539317,0.490189,0.400403,0.640617,0.444488,0.689128,0.554662
2,0.513232,0.446017,0.495208,0.567542,0.454211,0.465389,0.469724,0.587273,0.527634
3,0.580332,0.460637,0.378786,0.750484,0.683317,0.452301,0.500776,0.419377,0.435049
4,0.419914,0.787448,0.520750,0.410700,0.592801,0.595177,0.441963,0.550796,0.411871
5,0.596181,0.358905,0.423155,0.770292,0.412165,0.479685,0.519802,0.475791,0.641198
6,0.405944,0.918329,0.613290,0.395366,0.587756,0.507267,0.429304,0.542333,0.412694
7,0.412245,0.549312,0.793756,0.399818,0.444218,0.546397,0.441389,0.527451,0.543935
8,0.464467,0.544239,0.475684,0.478198,0.502018,0.636343,0.467384,0.736197,0.415882
9,0.529916,0.400516,0.474880,0.638111,0.428739,0.549056,0.474324,0.493503,0.555179


In [31]:
from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(n_components=9, covariance_type="full", random_state=42)
df_model["gmm_role"] = gmm.fit_predict(X_pca)
pd.crosstab(df_model["role_id"], df_model["gmm_role"])

gmm_role,0,1,2,3,4,5,6,7,8
role_id,,,,,,,,,
0,58,0,0,0,0,0,6,1,0
1,0,86,1,0,1,1,0,2,0
2,0,0,91,0,0,0,0,1,1
3,25,0,0,159,0,11,1,0,0
4,0,1,0,5,91,0,0,0,0
5,0,2,1,1,0,65,0,16,0
6,1,0,0,0,0,0,29,0,0
7,5,1,2,52,0,12,0,91,0
8,0,0,0,23,0,1,0,0,106


In [32]:
from sklearn.model_selection import train_test_split

X1, X2 = train_test_split(X_pca, test_size=0.5, random_state=42)

k1 = KMeans(9).fit(X1)
k2 = KMeans(9).fit(X2)

from sklearn.metrics import adjusted_rand_score
adjusted_rand_score(k1.labels_, k2.predict(X1))


C:\Users\enmat\anaconda3\envs\football-analytics-ml\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(
C:\Users\enmat\anaconda3\envs\football-analytics-ml\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(


0.6363414584866438

In [38]:
df_players = df1[["player_key", "player", "player_position"]].copy()
df_model["player_key"] = df1["player_key"].values
df_model["player_key"] = df1["player_key"].values
df_model = df_model.merge(df_players, on="player_key", how="left")


In [39]:
df_model.groupby("role_id")["player_position"].value_counts(normalize=True)


role_id  player_position          
0        Center Forward               0.313953
         Left Wing                    0.093023
         Right Center Forward         0.093023
         Right Wing                   0.093023
         Right Midfield               0.081395
                                        ...   
8        Left Back                    0.031646
         Center Defensive Midfield    0.012658
         Right Center Midfield        0.012658
         Left Wing                    0.006329
         Right Defensive Midfield     0.006329
Name: proportion, Length: 103, dtype: float64

In [40]:
df_model.groupby("role_id")["player"].head(10)


0                   Ismaïla Sarr
1                   Ismaïla Sarr
2                Youri Tielemans
3                Youri Tielemans
4             Saîf-Eddine Khaoui
                 ...            
169    Yasir Gharsan Al Shahrani
183                 Artem Dzyuba
215                Sardar Azmoun
216                Sardar Azmoun
250                Marwan Mohsen
Name: player, Length: 90, dtype: object

In [41]:
df_model[["player","player_position","role_id"]].head(15)


,player,player_position,role_id
0,Ismaïla Sarr,Right Wing,8
1,Ismaïla Sarr,Left Wing,8
2,Youri Tielemans,Left Center Midfield,7
3,Youri Tielemans,Right Defensive Midfield,7
4,Saîf-Eddine Khaoui,Left Center Midfield,7
5,Ángel Fabián Di María Hernández,Left Wing,3
6,Ángel Fabián Di María Hernández,Right Midfield,3
7,Presnel Kimpembe,Left Center Back,1
8,Kylian Mbappé Lottin,Right Wing,3
9,Kylian Mbappé Lottin,Left Wing,3


In [74]:
player_role_stability = (
    df_model
    .groupby("player")["role_id"]
    .nunique()
    .sort_values()
)

player_role_stability.sample(100)

player
Francisco Román Alarcón Suárez    1
Virgil van Dijk                   1
Hotaru Yamaguchi                  1
Mikael Lustig                     1
Hussain Al Mogahwi                1
                                 ..
Steven Zuber                      1
Rasmus Nissen Kristensen          1
Bassam Hisham Al Rawi             1
Duje Ćaleta-Car                   1
Marcos Danilo Ureña Porras        1
Name: role_id, Length: 100, dtype: int64

In [52]:
from sklearn.metrics.pairwise import cosine_similarity

role_centroids = X_std.groupby("role_id")[FEATURES].mean()

player_role_profile = pd.DataFrame(
    cosine_similarity(X_std[FEATURES], role_centroids),
    index=df_players["player"],
    columns=[f"role_{i}" for i in role_centroids.index]
)
player_role_profile

,role_0,role_1,role_2,role_3,role_4,role_5,role_6,role_7,role_8
player,,,,,,,,,
Ismaïla Sarr,0.315339,-0.618703,-0.149744,0.453263,-0.420672,-0.414472,0.276794,-0.299006,0.659244
Youri Tielemans,-0.028106,-0.247913,0.096454,-0.057893,-0.448999,0.364887,-0.084883,0.425537,0.149988
Saîf-Eddine Khaoui,0.202712,-0.240787,0.004329,0.165272,-0.285230,-0.120194,0.069390,0.216787,0.112757
Ángel Fabián Di María Hernández,0.469382,-0.211956,-0.624305,0.665688,0.489254,-0.249151,0.194441,-0.453898,-0.277398
Presnel Kimpembe,-0.387712,0.675559,0.128897,-0.451509,0.222161,0.338000,-0.303999,0.194245,-0.403094
...,...,...,...,...,...,...,...,...,...
Jewison Bennette,0.317042,-0.192580,-0.248154,0.425560,0.001758,-0.393899,0.292784,-0.215487,0.046239
Pablo Martín Páez Gavira,0.065409,-0.425394,-0.011144,0.133430,-0.478864,0.069192,0.180912,0.304132,0.319695
Achraf Dari,-0.701067,0.608739,0.856330,-0.734671,-0.156731,0.024376,-0.535763,0.043058,0.094936


In [45]:
role_profiles = df_model.groupby("role_id")[FEATURES].mean()
role_profiles

,passes_per_90,short_passes_per_90,medium_passes_per_90,long_passes_per_90,crosses_per90,switches_per90,throughballs_per90,cutbacks_per90,backheels_per90,passes_under_pressure_per90,...,pass_angle_mean_mid_third,pass_angle_var_mid_third,pass_angle_mean_att_third,pass_angle_var_att_third,pass_angle_mean_left_channel,pass_angle_var_left_channel,pass_angle_mean_centre_channel,pass_angle_var_centre_channel,pass_angle_mean_right_channel,pass_angle_var_right_channel
role_id,,,,,,,,,,,,,,,,,,,,,
0,28.445801,14.960284,9.348818,4.136700,1.179562,0.991784,0.150113,0.072165,0.316421,7.080778,...,19.355757,0.745155,0.216018,0.758226,69.368919,0.693127,27.892550,0.715632,-59.895362,0.633278
1,49.570321,14.575744,23.796852,11.197724,0.111779,1.620115,0.041407,0.006238,0.011389,5.160585,...,20.690383,0.550333,15.567488,0.304608,34.377870,0.499632,8.736452,0.562114,-13.535922,0.321572
2,50.080576,13.516439,24.126510,12.437626,0.154837,1.883050,0.090897,0.000000,0.012071,5.648489,...,-29.931092,0.569372,-25.023242,0.363040,9.215615,0.396856,-18.491307,0.593719,-37.911282,0.515710
3,29.037314,15.473004,9.638899,3.925411,1.709317,0.927219,0.111691,0.109959,0.247317,6.687785,...,13.617638,0.690986,-1.290508,0.727420,69.164895,0.599984,5.754913,0.643096,-63.313031,0.574989
4,47.217446,21.419184,18.543633,7.254628,2.177117,1.057051,0.067397,0.094273,0.096383,7.463592,...,74.351286,0.546399,87.718413,0.456129,73.619744,0.506396,27.384075,0.453846,-28.943086,0.092529
5,74.234629,31.042389,30.990576,12.201664,1.080536,2.489537,0.246814,0.056657,0.188018,11.922069,...,8.256211,0.785452,6.620053,0.758797,46.629379,0.715930,4.588523,0.756325,-40.789745,0.712978
6,18.822854,12.136388,5.293129,1.393338,0.551240,0.259742,0.037703,0.160304,0.174116,5.368631,...,-49.113477,0.701541,25.695115,0.680979,51.140230,0.528009,-6.895497,0.614930,-59.265179,0.618263
7,39.516944,17.251071,15.780968,6.484905,0.464571,1.144612,0.133011,0.037399,0.067202,7.223699,...,2.562753,0.791835,3.615534,0.684718,39.219505,0.699380,-10.456909,0.745346,-32.056034,0.707989
8,49.218742,21.379705,19.922058,7.916979,2.059380,1.053528,0.116850,0.110472,0.095310,7.440362,...,-76.775887,0.542859,-93.044288,0.453160,31.984603,0.096701,-33.531942,0.472301,-76.926466,0.499742


In [65]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

# player_role_profile: index=players, columns=role_0..role_8

sim = cosine_similarity(player_role_profile.values)  # shape: (n_players, n_players)

sim.shape



(950, 950)

In [66]:
sim

array([[ 1.        ,  0.08833099,  0.70409801, ..., -0.54546451,
        -0.86698981,  0.84524503],
       [ 0.08833099,  1.        ,  0.5508325 , ...,  0.12245363,
        -0.04045594,  0.10754447],
       [ 0.70409801,  0.5508325 ,  1.        , ..., -0.45473033,
        -0.63905345,  0.83138569],
       ...,
       [-0.54546451,  0.12245363, -0.45473033, ...,  1.        ,
         0.85878343, -0.41961051],
       [-0.86698981, -0.04045594, -0.63905345, ...,  0.85878343,
         1.        , -0.67809934],
       [ 0.84524503,  0.10754447,  0.83138569, ..., -0.41961051,
        -0.67809934,  1.        ]], shape=(950, 950))

In [67]:
def find_similar(player_name: str, top_n: int = 10):
    if player_name not in player_role_profile.index:
        raise KeyError(f"'{player_name}' not in player_role_profile index")

    idx = player_role_profile.index.get_loc(player_name)
    scores = sim[idx]  # length == n_players

    return (
        pd.Series(scores, index=player_role_profile.index, name="similarity")
        .sort_values(ascending=False)
        .iloc[1: top_n + 1]  # skip self
    )

find_similar("Kylian Mbappé Lottin", top_n=10)


ValueError: Length of values (2) does not match length of index (950)

In [62]:
print("player_role_profile:", player_role_profile.shape)
print("sim:", sim.shape)


player_role_profile: (950, 9)
sim: (950, 950)


In [63]:
np.diag(sim)[:10]


array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

In [64]:
sim.min(), sim.max()


(np.float64(-0.9956765362799848), np.float64(1.0000000000000004))

In [79]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

sim = cosine_similarity(player_role_profile.values)

def find_similar(player_name: str, sim_matrix, top_n: int = 10):
    prp = player_role_profile

    if player_name not in prp.index:
        raise KeyError(f"'{player_name}' not in player_role_profile index")

    idx = prp.index.get_loc(player_name)
    scores = sim_matrix[idx]

    return (
        pd.Series(scores, index=prp.index, name="similarity")
        .sort_values(ascending=False)
        .iloc[1: top_n + 1]
    )

find_similar("Gylfi Þór Sigurðsson", sim, top_n=20)


player
M''Baye Babacar Niang                      0.989107
Dušan Tadić                                0.981911
Adem Ljajić                                0.979887
Kamil Grosicki                             0.978757
Aleksandr Golovin                          0.974406
Leroy Sané                                 0.972180
Nicolai Jørgensen                          0.969823
Cody Mathès Gakpo                          0.964533
Ante Rebić                                 0.963148
Timo Werner                                0.962105
Giorgian Daniel De Arrascaeta Benedetti    0.960470
Gabriel Fernando de Jesus                  0.957984
Luis Alberto Suárez Díaz                   0.956883
Carlos Alberto Vela Garrido                0.956530
Heung-Min Son                              0.955367
Fahad Mosaed Al Muwallad Al Harbi          0.952810
Naïm Sliti                                 0.952375
Harry Kane                                 0.950672
Papa Alioune N''Diaye                      0.948886
Sergi

In [72]:
print(type(sim), getattr(sim, "shape", None))
print("Row length:", len(sim[0]))
print("Row length via idx:", len(sim[player_role_profile.index.get_loc("Kai Havert")]))


<class 'numpy.ndarray'> (950, 950)
Row length: 950
Row length via idx: 950
